# Project 2 · Convolutional Neural Network
**AI Workshop – From MLP to Transformers**

Networks that respect the shape of an image. We will:
1. **Train a CNN on MNIST**.
2. **Compare it with the MLP** from Project 1, including on shifted digits.
3. **Switch to CIFAR-10**: real colour photos, where the gap gets bigger.
4. **Visualize feature maps**: see what the kernels learned.

> **Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom (**Shift + Enter**).

## 0 · Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)              # 1 · forward pass
        loss = loss_fn(logits, labels)      # 2 · loss
        optimizer.zero_grad()
        loss.backward()                     # 3 · backpropagation
        optimizer.step()                    # 4 · update the weights
        loss_sum += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader, shift=0):
    """Accuracy on a dataset. shift > 0 moves every image that many pixels to the right."""
    model.eval()
    total, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if shift:
            images = torch.roll(images, shifts=shift, dims=3)
        correct += (model(images).argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return correct / total

def count_params(model):
    return sum(p.numel() for p in model.parameters())

## The CNN
Conv → ReLU → Pool, twice, then a small MLP on top: the architecture from the slides.
`padding=1` keeps the size the same, and each `MaxPool2d(2)` halves it.

> ✏️ **Your turn:** replace every `TODO` in the cell below, then run it. Stuck? The solution is in the **Solutions** section at the end of the notebook.

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_channels=1, image_size=28, num_classes=10):
        super().__init__()
        # ✏️ Fill in: Conv → ReLU → Pool, twice
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=TODO, padding=TODO), nn.ReLU(),  # 32 kernels of 3 × 3, keep the size
            TODO,                                                                 # halve width and height
            nn.Conv2d(TODO, 64, kernel_size=3, padding=1), TODO,                   # how many channels come in? then ReLU
            nn.MaxPool2d(2),
        )
        flat = 64 * (image_size // 4) ** 2        # MNIST: 64 × 7 × 7 = 3,136 · CIFAR-10: 64 × 8 × 8 = 4,096
        # ✏️ Fill in: a small MLP on top
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(TODO, 128), nn.ReLU(),       # how many numbers come out of Flatten?
            nn.Linear(128, TODO),                  # one score per class
        )

    def forward(self, x):
        return self.classifier(self.features(x))

The MLP from Project 1, for comparison (ready to run).

In [ ]:
class MLP(nn.Module):
    """The Project 1 model, generalised to any input size."""
    def __init__(self, in_features=784, hidden=(128, 64), num_classes=10):
        super().__init__()
        layers, size = [nn.Flatten()], in_features
        for h in hidden:
            layers += [nn.Linear(size, h), nn.ReLU()]
            size = h
        layers.append(nn.Linear(size, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

Follow one CIFAR-10 image through the network and watch the shape change, exactly like the architecture slide.

In [ ]:
demo = CNN(in_channels=3, image_size=32)
x = torch.zeros(1, 3, 32, 32)
print(f"{'Input':<10} → {tuple(x.shape[1:])}")
for layer in list(demo.features) + list(demo.classifier):
    x = layer(x)
    print(f"{layer.__class__.__name__:<10} → {tuple(x.shape[1:])}")

## 01 · Train on MNIST
We use **Adam**, a smarter version of gradient descent that adapts the step size for every weight. The four training steps stay exactly the same.

In [ ]:
mnist_tf = transforms.ToTensor()
mnist_train = datasets.MNIST("data", train=True, download=True, transform=mnist_tf)
mnist_test = datasets.MNIST("data", train=False, download=True, transform=mnist_tf)
mnist_train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
mnist_test_loader = DataLoader(mnist_test, batch_size=256)

EPOCHS_MNIST = 3
cnn_mnist = CNN(in_channels=1, image_size=28).to(device)
opt = torch.optim.Adam(cnn_mnist.parameters(), lr=1e-3)
for epoch in range(1, EPOCHS_MNIST + 1):
    loss, acc = train_one_epoch(cnn_mnist, mnist_train_loader, opt)
    print(f"Epoch {epoch}: loss {loss:.3f} · train acc {acc:.1%} · test acc {evaluate(cnn_mnist, mnist_test_loader):.1%}")

## 02 · Compare with the MLP
Same data, same epochs, same optimizer: only the model changes.

In [ ]:
mlp_mnist = MLP(in_features=784, hidden=(128, 64)).to(device)
opt = torch.optim.Adam(mlp_mnist.parameters(), lr=1e-3)
for epoch in range(1, EPOCHS_MNIST + 1):
    loss, acc = train_one_epoch(mlp_mnist, mnist_train_loader, opt)
    print(f"MLP epoch {epoch}: loss {loss:.3f} · train acc {acc:.1%}")

print()
print(f"{'model':<6}{'parameters':>12}{'test accuracy':>16}")
for name, m in [("MLP", mlp_mnist), ("CNN", cnn_mnist)]:
    print(f"{name:<6}{count_params(m):>12,}{evaluate(m, mnist_test_loader):>16.2%}")

### Experiment: shift the digits
MNIST digits are always centred. What happens if we move every test digit a few pixels to the right?
The CNN's kernels look for the same patterns everywhere (**weight sharing**), but its final dense layer still cares about position, so expect both models to drop. Which one holds up better?

In [ ]:
shifts = [0, 2, 4, 6]
print(f"{'shift (px)':<12}{'MLP':>8}{'CNN':>8}")
for s in shifts:
    print(f"{s:<12}{evaluate(mlp_mnist, mnist_test_loader, shift=s):>8.1%}{evaluate(cnn_mnist, mnist_test_loader, shift=s):>8.1%}")

img = mnist_test[0][0]
fig, axes = plt.subplots(1, len(shifts), figsize=(8, 2.2))
for ax, s in zip(axes, shifts):
    ax.imshow(torch.roll(img, shifts=s, dims=2).squeeze(), cmap="gray"); ax.set_title(f"shift {s}"); ax.axis("off")
plt.show()

## 03 · Switch to CIFAR-10
32 × 32 **colour** photos in 10 classes. Now each image is 32 × 32 × 3 = 3,072 numbers.
We keep training short (3 epochs) so everyone finishes in time; the gap between MLP and CNN is already visible.

In [ ]:
CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

cifar_tf = transforms.Compose([
    transforms.ToTensor(),                                   # 0–255 → 0–1
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # → −1 to 1 per channel
])
cifar_train = datasets.CIFAR10("data", train=True, download=True, transform=cifar_tf)
cifar_test = datasets.CIFAR10("data", train=False, download=True, transform=cifar_tf)
cifar_train_loader = DataLoader(cifar_train, batch_size=64, shuffle=True, num_workers=2)
cifar_test_loader = DataLoader(cifar_test, batch_size=256, num_workers=2)

def show(img):  # undo the normalisation for plotting
    return (img.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    img, lab = cifar_train[i]
    ax.imshow(show(img)); ax.set_title(CLASSES[lab]); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
EPOCHS_CIFAR = 3

cnn_cifar = CNN(in_channels=3, image_size=32).to(device)
opt = torch.optim.Adam(cnn_cifar.parameters(), lr=1e-3)
for epoch in range(1, EPOCHS_CIFAR + 1):
    loss, acc = train_one_epoch(cnn_cifar, cifar_train_loader, opt)
    print(f"CNN epoch {epoch}: loss {loss:.3f} · train acc {acc:.1%} · test acc {evaluate(cnn_cifar, cifar_test_loader):.1%}")

# the MLP from the slides: 3,072 inputs → 512 → 128 → 10
mlp_cifar = MLP(in_features=3 * 32 * 32, hidden=(512, 128)).to(device)
opt = torch.optim.Adam(mlp_cifar.parameters(), lr=1e-3)
for epoch in range(1, EPOCHS_CIFAR + 1):
    loss, acc = train_one_epoch(mlp_cifar, cifar_train_loader, opt)
    print(f"MLP epoch {epoch}: loss {loss:.3f} · train acc {acc:.1%} · test acc {evaluate(mlp_cifar, cifar_test_loader):.1%}")

print()
print(f"{'model':<6}{'parameters':>12}{'test accuracy':>16}")
for name, m in [("MLP", mlp_cifar), ("CNN", cnn_cifar)]:
    print(f"{name:<6}{count_params(m):>12,}{evaluate(m, cifar_test_loader):>16.2%}")
print("(random guessing would be 10%)")

In [ ]:
images, labels = next(iter(cifar_test_loader))
with torch.no_grad():
    preds = cnn_cifar(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.8))
for i, ax in enumerate(axes.flat):
    ax.imshow(show(images[i]))
    ax.set_title(CLASSES[preds[i]], color="green" if preds[i] == labels[i] else "red")
    ax.axis("off")
plt.suptitle("CNN predictions: green = correct, red = wrong")
plt.tight_layout()
plt.show()

## 04 · Visualize feature maps
First, the 32 kernels of the first layer: each is a 3 × 3 × 3 patch of weights, shown as a tiny colour image.
Then the **feature maps** they produce for one photo: bright where the kernel's pattern shows up.

In [ ]:
kernels = cnn_cifar.features[0].weight.detach().cpu()        # 32 × 3 × 3 × 3
kernels = (kernels - kernels.min()) / (kernels.max() - kernels.min())
fig, axes = plt.subplots(4, 8, figsize=(8, 4))
for k, ax in zip(kernels, axes.flat):
    ax.imshow(k.permute(1, 2, 0)); ax.axis("off")
plt.suptitle("First-layer kernels (3 × 3 × 3 each)")
plt.show()

In [ ]:
img, lab = cifar_test[7]
x = img.unsqueeze(0).to(device)
with torch.no_grad():
    maps1 = cnn_cifar.features[:2](x)[0].cpu()     # conv 1 + ReLU → 32 × 32 × 32
    maps2 = cnn_cifar.features[:5](x)[0].cpu()     # … conv 2 + ReLU → 64 × 16 × 16

plt.figure(figsize=(2.5, 2.5)); plt.imshow(show(img)); plt.title(CLASSES[lab]); plt.axis("off"); plt.show()

for maps, title in [(maps1, "Layer 1 feature maps: edges and colours"), (maps2, "Layer 2 feature maps: textures and shapes")]:
    fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
    for m, ax in zip(maps, axes.flat):
        ax.imshow(m, cmap="viridis"); ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

### Try it yourself
- Train for more epochs (`EPOCHS_CIFAR = 10`): how high does the CNN get?
- Add a third conv block with 128 kernels.
- Change `cifar_test[7]` to look at the feature maps of other photos.

## 🔑 Solutions
Only if you are stuck: run the solution for your step (double-click to see the code), then go back and continue where you were.

In [ ]:
# @title Solution · The CNN
class CNN(nn.Module):
    def __init__(self, in_channels=1, image_size=28, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1), nn.ReLU(),  # 32 kernels of 3 × 3
            nn.MaxPool2d(2),                                                   # halve width and height
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),           # 64 kernels
            nn.MaxPool2d(2),
        )
        flat = 64 * (image_size // 4) ** 2        # MNIST: 64 × 7 × 7 = 3,136 · CIFAR-10: 64 × 8 × 8 = 4,096
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))